# Chapter 20 — Endogeneity
## University of Neuchâtel — Python for Econometrics

---

## What you will learn

| Section | Topic |
|---|---|
| 1 | What is endogeneity? |
| 2 | Source 1: Omitted Variable Bias |
| 3 | Source 2: Simultaneity |
| 4 | Source 3: Measurement Error |
| 5 | Direction of bias: analytical formula |
| 6 | Detection: Durbin-Wu-Hausman test |
| 7 | Solutions overview |

**Format:** One cell per step. Manual calculation first, statsmodels second, `np.isclose()` verification.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import wooldridge as woo
import os
try:
    from linearmodels.iv import IV2SLS
    HAS_LM = True
except ImportError:
    HAS_LM = False

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

---
## 1 — What is Endogeneity?

### 1.1 The Core OLS Assumption

OLS relies on the **zero conditional mean** assumption:

$$E(u \mid x_1, x_2, ..., x_k) = 0$$

This means: **the error is uncorrelated with all regressors**.

When this fails — when $Cov(x_j, u) \neq 0$ for some $j$ — we say $x_j$ is **endogenous**.

### 1.2 Consequence

If $x_j$ is endogenous, OLS is **biased and inconsistent**:

$$\hat{\beta}_{OLS} \xrightarrow{p} \beta + \frac{Cov(x, u)}{Var(x)} \neq \beta$$

Even with infinite data, OLS does not converge to the truth. This is the most serious
problem in applied econometrics because it invalidates all inference.

### 1.3 Three Sources of Endogeneity

| Source | Mechanism | Example |
|---|---|---|
| **Omitted variable bias** | Unmeasured variable correlated with $x$ and $y$ | Ability omitted from wage equation |
| **Simultaneity** | $x$ and $y$ jointly determined | Price and quantity |
| **Measurement error** | $x$ is measured with error | Self-reported wages |

### 1.4 Intuition: Why OLS Absorbs the Omitted Variable

True model: $y = \beta_0 + \beta_1 x + \gamma a + u$ (where $a$ = omitted variable)

What we estimate: $y = \tilde{\beta}_0 + \tilde{\beta}_1 x + \tilde{u}$ where $\tilde{u} = \gamma a + u$

If $Cov(x, a) \neq 0$, then $Cov(x, \tilde{u}) \neq 0$ → OLS is biased.

---
## 2 — Source 1: Omitted Variable Bias (OVB)

### 2.1 The OVB Formula

Suppose the true model is:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + u$$

But we **omit** $x_2$ and estimate:

$$y = \tilde{\beta}_0 + \tilde{\beta}_1 x_1 + \tilde{u}$$

The OLS estimator of $\tilde{\beta}_1$ converges to:

$$\text{plim}(\tilde{\beta}_1) = \beta_1 + \beta_2 \cdot \delta_1$$

where $\delta_1 = \frac{Cov(x_1, x_2)}{Var(x_1)}$ is the slope from regressing $x_2$ on $x_1$.

**Bias** = $\beta_2 \cdot \delta_1$

### 2.2 Direction of the Bias

| $\beta_2$ (effect of omitted on y) | $\delta_1$ (correlation of omitted with x) | Bias direction |
|---|---|---|
| $+$ | $+$ | **Upward** (OLS overstates $\beta_1$) |
| $+$ | $-$ | **Downward** |
| $-$ | $+$ | **Downward** |
| $-$ | $-$ | **Upward** |

### Step 2.1 — Simulate OVB

In [ ]:
np.random.seed(42)
n = 1000
true_b0 = 1.0
true_b1 = 0.5   # true effect of educ
true_b2 = 0.8   # true effect of ability (omitted)
# ability and educ are positively correlated
cov_mat = np.array([[1, 0.6],[0.6, 1]])
Xdata   = np.random.multivariate_normal([12, 100], cov_mat*np.diag([4,100]), n)
educ, ability = Xdata[:,0], Xdata[:,1]
u    = np.random.normal(0, 1, n)
lwage = true_b0 + true_b1*educ + true_b2*ability + u
print(f'True b1(educ):    {true_b1}')
print(f'True b2(ability): {true_b2}')
print(f'Corr(educ,ability): {np.corrcoef(educ,ability)[0,1]:.3f}')

### Step 2.2 — Short regression (omits ability) — biased

In [ ]:
b1_short_manual = np.sum((educ-educ.mean())*(lwage-lwage.mean())) / np.sum((educ-educ.mean())**2)
print(f'Short OLS b1 (manual): {b1_short_manual:.4f}')

In [ ]:
df_sim = pd.DataFrame({'lwage': lwage, 'educ': educ, 'ability': ability})
res_short = smf.ols('lwage ~ educ', data=df_sim).fit()
print(f'Short OLS b1 (statsmodels): {res_short.params["educ"]:.4f}')
print(f'Match: {np.isclose(b1_short_manual, res_short.params["educ"])}')

### Step 2.3 — Long regression (includes ability) — unbiased

In [ ]:
res_long = smf.ols('lwage ~ educ + ability', data=df_sim).fit()
print(f'Long OLS b1: {res_long.params["educ"]:.4f}  (true = {true_b1})')
print(f'Long OLS b2: {res_long.params["ability"]:.4f}  (true = {true_b2})')

### Step 2.4 — Verify OVB formula analytically

In [ ]:
# OVB formula: plim(b1_short) = b1 + b2 * delta1
# where delta1 = Cov(educ, ability) / Var(educ)
delta1 = np.cov(educ, ability)[0,1] / np.var(educ, ddof=1)
ovb_formula = true_b1 + true_b2 * delta1
print(f'OVB formula: b1 + b2*delta1 = {true_b1} + {true_b2:.4f}*{delta1:.4f} = {ovb_formula:.4f}')
print(f'Short OLS b1:                  {res_short.params["educ"]:.4f}')
print(f'Match (formula vs OLS): {np.isclose(ovb_formula, res_short.params["educ"], rtol=0.05)}')
print(f'Upward bias because: b2>0 AND Corr(educ,ability)>0')

### Step 2.5 — Visualise: how bias varies with sample size

In [ ]:
sample_sizes = [50, 100, 200, 500, 1000, 5000]
b1_short_means = []
b1_long_means  = []
np.random.seed(7)
for nn in sample_sizes:
    reps = []
    repl = []
    for _ in range(500):
        Xd   = np.random.multivariate_normal([12,100], cov_mat*np.diag([4,100]), nn)
        ed, ab = Xd[:,0], Xd[:,1]
        lw = true_b0 + true_b1*ed + true_b2*ab + np.random.normal(0,1,nn)
        b_s = np.sum((ed-ed.mean())*(lw-lw.mean())) / np.sum((ed-ed.mean())**2)
        X_l = np.column_stack([np.ones(nn), ed, ab])
        b_l = np.linalg.solve(X_l.T@X_l, X_l.T@lw)[1]
        reps.append(b_s)
        repl.append(b_l)
    b1_short_means.append(np.mean(reps))
    b1_long_means.append(np.mean(repl))
fig, ax = plt.subplots(figsize=(9,4))
ax.semilogx(sample_sizes, b1_short_means, 'ro-', label='Short (omits ability) — biased', linewidth=2)
ax.semilogx(sample_sizes, b1_long_means,  'bs-', label='Long (includes ability) — unbiased', linewidth=2)
ax.axhline(true_b1, color='black', linestyle='--', linewidth=2, label=f'True b1={true_b1}')
ax.set_xlabel('Sample size (log scale)')
ax.set_ylabel('Mean b1_hat (500 replications)')
ax.set_title('OVB persists with large samples — inconsistency')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch20_01_ovb_consistency.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: short regression bias does NOT shrink with more data (inconsistent)')

---
## 3 — Source 2: Simultaneity Bias

### 3.1 Mechanism

When $y$ and $x$ are **jointly determined**, regressing $y$ on $x$ gives biased estimates
in both directions — neither equation's coefficients are identified by simple OLS.

**Classic example: Supply and Demand**

$$Q = \alpha_0 + \alpha_1 P + u_d \quad \text{(demand)}$$
$$Q = \beta_0 + \beta_1 P + u_s \quad \text{(supply)}$$

Equilibrium price $P$ is determined by **both** equations simultaneously.
So $P$ is correlated with both $u_d$ and $u_s$.

### 3.2 Direction of Simultaneity Bias

For the demand equation, OLS plim:

$$\text{plim}(\hat{\alpha}_1^{OLS}) = \frac{\alpha_1 \sigma_s^2 + \beta_1 \sigma_d^2}{\sigma_s^2 + \sigma_d^2}$$

This is a **weighted average** of the demand and supply slopes — not the demand slope alone.
OLS identifies neither curve when both shift.

### Step 3.1 — Simulate simultaneity bias

In [ ]:
np.random.seed(1)
n = 800
true_alpha1 = -2.0   # demand slope (negative)
true_beta1  =  1.5   # supply slope (positive)
sigma_d, sigma_s = 3.0, 2.0
ud = np.random.normal(0, sigma_d, n)
us = np.random.normal(0, sigma_s, n)
# Equilibrium: alpha1*P + ud = beta1*P + us
# P = (us - ud) / (alpha1 - beta1)
P = (us - ud) / (true_alpha1 - true_beta1)
Q = true_alpha1*P + ud  # = true_beta1*P + us
df_sd = pd.DataFrame({'Q': Q, 'P': P})
print(f'True demand slope: {true_alpha1}')
print(f'True supply slope: {true_beta1}')

In [ ]:
# Analytical formula for simultaneity bias
plim_ols = (true_alpha1*sigma_s**2 + true_beta1*sigma_d**2) / (sigma_s**2 + sigma_d**2)
print(f'plim(OLS slope) = {plim_ols:.4f}  (neither {true_alpha1} nor {true_beta1})')

In [ ]:
res_sim_ols = smf.ols('Q ~ P', data=df_sd).fit()
ols_slope = res_sim_ols.params['P']
print(f'OLS slope (estimate):    {ols_slope:.4f}')
print(f'Analytical bias formula: {plim_ols:.4f}')
print(f'True demand slope:       {true_alpha1}')
print(f'True supply slope:       {true_beta1}')
print(f'Match (OLS vs formula): {np.isclose(ols_slope, plim_ols, rtol=0.05)}')

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(P, Q, alpha=0.15, s=10, label='Equilibrium data')
p_range = np.linspace(P.min(), P.max(), 100)
ax.plot(p_range, res_sim_ols.params['Intercept'] + ols_slope*p_range,
        'r-', linewidth=2.5, label=f'OLS: slope={ols_slope:.2f} (biased)')
ax.set_xlabel('Price')
ax.set_ylabel('Quantity')
ax.set_title('Simultaneity: OLS identifies neither supply nor demand')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch20_02_simultaneity.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Source 3: Measurement Error

### 4.1 Classical Measurement Error (Errors-in-Variables)

Suppose the true regressor is $x^*$ but we only observe a noisy version:

$$x = x^* + v, \quad v \sim (0, \sigma_v^2), \quad Cov(x^*, v) = 0$$

We estimate $y = \beta_0 + \beta_1 x + u$ instead of $y = \beta_0 + \beta_1 x^* + u$.

### 4.2 Attenuation Bias Formula

OLS converges to:

$$\text{plim}(\hat{\beta}_1) = \beta_1 \cdot \underbrace{\frac{\sigma_{x^*}^2}{\sigma_{x^*}^2 + \sigma_v^2}}_{\text{reliability ratio} \in (0,1)}$$

The OLS estimate is **attenuated toward zero** (downward in absolute value).
The larger the measurement error variance $\sigma_v^2$ relative to $\sigma_{x^*}^2$,
the more the coefficient is biased toward zero.

**Important:** Measurement error in the **dependent variable** $y$ only inflates the error
variance but does NOT cause bias in OLS.

### Step 4.1 — Simulate attenuation bias

In [ ]:
np.random.seed(3)
n = 2000
true_b1 = 2.0
x_star   = np.random.normal(0, 2, n)      # true regressor
u_true   = np.random.normal(0, 1, n)
y_me     = 1 + true_b1*x_star + u_true
print(f'True b1 = {true_b1}')

In [ ]:
results_me = []
for sigma_v in [0, 0.5, 1.0, 2.0, 4.0]:
    v = np.random.normal(0, sigma_v, n)
    x_obs = x_star + v  # observed (mismeasured) x
    b1_hat = np.sum((x_obs-x_obs.mean())*(y_me-y_me.mean())) / np.sum((x_obs-x_obs.mean())**2)
    # Reliability ratio
    rel_ratio = np.var(x_star,ddof=1) / (np.var(x_star,ddof=1) + sigma_v**2)
    b1_formula = true_b1 * rel_ratio
    results_me.append({'sigma_v': sigma_v, 'b1_hat': b1_hat,
                       'b1_formula': b1_formula, 'reliability': rel_ratio})
df_me = pd.DataFrame(results_me)
print(df_me.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(df_me['sigma_v'], df_me['b1_hat'],    'ro-', linewidth=2, label='OLS estimate')
ax.plot(df_me['sigma_v'], df_me['b1_formula'],'bs--',linewidth=2, label='Formula: b1*reliability')
ax.axhline(true_b1, color='black', linestyle='--', linewidth=2, label=f'True b1={true_b1}')
ax.axhline(0,        color='gray',  linestyle=':',  linewidth=1)
ax.set_xlabel('Measurement error std (sigma_v)')
ax.set_ylabel('b1 estimate')
ax.set_title('Attenuation Bias: coefficient shrinks toward 0 as measurement error grows')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch20_03_attenuation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Formula matches OLS estimate:', np.allclose(df_me.b1_hat, df_me.b1_formula, rtol=0.05))

---
## 5 — Real Data: Returns to Education (OVB from Ability)

We use the `wage2` dataset to quantify ability bias in the return to education.

In [ ]:
df = woo.dataWoo('wage2')
df = df[['lwage','educ','IQ','exper','tenure','married','black']].dropna()
print('Shape:', df.shape)
print(df[['lwage','educ','IQ','exper']].describe().round(2))

### Step 5.1 — Short regression (omits IQ — proxy for ability)

In [ ]:
n = len(df)
X_short = np.column_stack([np.ones(n), df['educ'].values, df['exper'].values,
                           df['tenure'].values, df['married'].values, df['black'].values])
y_arr = df['lwage'].values
b_short = np.linalg.solve(X_short.T@X_short, X_short.T@y_arr)
print(f'Short model b_educ (manual): {b_short[1]:.4f}')

In [ ]:
res_short_real = smf.ols('lwage ~ educ + exper + tenure + married + black', data=df).fit()
print(f'Short model b_educ (sm):     {res_short_real.params["educ"]:.4f}')
print(f'Match: {np.isclose(b_short[1], res_short_real.params["educ"])}')

### Step 5.2 — Long regression (adds IQ as proxy for ability)

In [ ]:
X_long = np.column_stack([np.ones(n), df['educ'].values, df['IQ'].values,
                          df['exper'].values, df['tenure'].values,
                          df['married'].values, df['black'].values])
b_long = np.linalg.solve(X_long.T@X_long, X_long.T@y_arr)
print(f'Long model b_educ (manual): {b_long[1]:.4f}')

In [ ]:
res_long_real = smf.ols('lwage ~ educ + IQ + exper + tenure + married + black', data=df).fit()
print(f'Long model b_educ (sm):     {res_long_real.params["educ"]:.4f}')
print(f'Match: {np.isclose(b_long[1], res_long_real.params["educ"])}')

### Step 5.3 — Quantify the ability bias

In [ ]:
bias_est = res_short_real.params['educ'] - res_long_real.params['educ']
# OVB formula: bias ≈ b_IQ * delta1  where delta1 = Cov(educ,IQ)/Var(educ)
delta1_real = np.cov(df['educ'], df['IQ'])[0,1] / df['educ'].var(ddof=1)
b_IQ = res_long_real.params['IQ']
bias_formula = b_IQ * delta1_real
print(f'Observed bias (short - long): {bias_est:.4f}')
print(f'OVB formula (b_IQ * delta1):  {bias_formula:.4f}')
print(f'Match: {np.isclose(bias_est, bias_formula, rtol=0.05)}')
print(f'\nInterpretation: ~{100*bias_est:.1f}pp of the educ coefficient')
print('was due to omitted ability — OLS overstated the return to education')

---
## 6 — Detection: Durbin-Wu-Hausman (DWH) Test

### 6.1 What the test does

The DWH test compares OLS (efficient under $H_0$) with IV/2SLS (consistent under both):

**$H_0$:** $x$ is exogenous — $Cov(x, u) = 0$ → OLS is consistent

**$H_1$:** $x$ is endogenous — $Cov(x, u) \neq 0$ → OLS is inconsistent

### 6.2 Control Function Approach (manual)

1. Run **first stage:** regress the suspected endogenous variable $x$ on all exogenous variables including instrument $z$
2. Get first-stage residuals $\hat{v}$
3. Include $\hat{v}$ as an extra regressor in the **main equation**
4. **t-test on $\hat{v}$:** significant → $x$ is endogenous

Intuition: $\hat{v}$ captures the part of $x$ correlated with $u$.
If $\hat{v}$ is significant, that correlation exists.

### Step 6.1 — Load card data (college proximity instrument)

In [ ]:
card = woo.dataWoo('card')
card = card.dropna(subset=['lwage','educ','nearc4','exper','black','smsa'])
print('card dataset shape:', card.shape)
print(card[['lwage','educ','nearc4','exper','black']].head(8))

### Step 6.2 — Stage 1: educ ~ nearc4 + exper + black + smsa

In [ ]:
res_s1 = smf.ols('educ ~ nearc4 + exper + black + smsa', data=card).fit()
v_hat  = res_s1.resid
card   = card.copy()
card['v_hat'] = v_hat.values
print('Stage 1 results:')
print(res_s1.params.round(4))
print(f'\nF-stat on nearc4 (instrument strength): {res_s1.f_test("nearc4=0").fvalue[0][0]:.2f}')

### Step 6.3 — Stage 2 augmented: include v_hat

In [ ]:
res_dwh = smf.ols('lwage ~ educ + exper + black + smsa + v_hat', data=card).fit()
t_vhat = res_dwh.tvalues['v_hat']
p_vhat = res_dwh.pvalues['v_hat']
print('DWH test (control function):')
print(f'  Coefficient on v_hat: {res_dwh.params["v_hat"]:.4f}')
print(f'  t-statistic:          {t_vhat:.4f}')
print(f'  p-value:              {p_vhat:.4f}')
if p_vhat < 0.05:
    print('  Reject H0: educ is ENDOGENOUS → use IV/2SLS')
else:
    print('  Fail to reject H0: no strong evidence of endogeneity')

### Step 6.4 — Compare OLS vs 2SLS

In [ ]:
res_ols_card = smf.ols('lwage ~ educ + exper + black + smsa', data=card).fit()
n_c = len(card)
# Manual 2SLS
educ_hat = res_s1.fittedvalues.values
X_s2 = np.column_stack([np.ones(n_c), educ_hat,
                        card['exper'].values, card['black'].values, card['smsa'].values])
y_c  = card['lwage'].values
b_2sls = np.linalg.solve(X_s2.T@X_s2, X_s2.T@y_c)
print('Comparison: OLS vs 2SLS')
print(f'  OLS  b_educ: {res_ols_card.params["educ"]:.4f}')
print(f'  2SLS b_educ: {b_2sls[1]:.4f}')
diff = b_2sls[1] - res_ols_card.params['educ']
print(f'  Difference:  {diff:+.4f}')
if diff < 0:
    print('  2SLS < OLS: consistent with upward ability bias in OLS')
else:
    print('  2SLS > OLS: LATE may differ from ATE, or weak instrument issues')

---
## 7 — Solutions to Endogeneity

| Solution | When | Requirement | Chapter |
|---|---|---|---|
| **Instrumental Variables (IV/2SLS)** | Any source of endogeneity | Valid instrument: relevant + exogenous | Ch 15 |
| **Fixed Effects (FE)** | Panel data, time-invariant endogeneity | Same individuals over time | Ch 14 |
| **Proxy variables** | Omitted variable bias | Observable proxy for omitted variable | Ch 9 |
| **RCT** | Policy evaluation | Random assignment of treatment | Ch 21 |
| **DiD** | Policy evaluation | Parallel trends assumption | Ch 13 |
| **Regression Discontinuity** | Cutoff-based assignment | Sharp discontinuity in treatment | Ch 21 |

### Step 7.1 — Summary: OLS bias across three sources

In [ ]:
summary = pd.DataFrame({
    'Source': ['Omitted Variable Bias', 'Simultaneity', 'Measurement Error'],
    'Cov(x,u)': ['≠ 0 (positive if corr>0)', '≠ 0 (both directions)', '≠ 0 (negative)'],
    'OLS bias': ['Upward or downward (sign of b2*delta1)', 'Weighted avg of slopes', 'Attenuation toward 0'],
    'Solution': ['IV, FE, proxy', 'IV/2SLS', 'IV with clean measure']
})
print(summary.to_string(index=False))
summary.to_csv(os.path.join(output_dir,'table_ch20_endogeneity_summary.csv'), index=False)
print('\nSaved table_ch20_endogeneity_summary.csv')

---
## 8 — Key Takeaways

1. **Endogeneity is a correlation problem:** $Cov(x_j, u) \neq 0$ means OLS is inconsistent — more data won't fix it.

2. **Three sources, one consequence:** OVB, simultaneity, and measurement error all violate $E(u|x)=0$ and all bias OLS.

3. **OVB formula is your friend:** $bias = \beta_{omitted} \times \delta$ tells you the *direction* and *magnitude* of the bias even before running any regression.

4. **Attenuation bias always goes toward zero:** measurement error in $x$ shrinks the coefficient. The more noise, the worse the attenuation.

5. **The DWH test detects endogeneity** — but you need a valid instrument for Stage 1. No instrument → can't run DWH properly.

6. **Endogeneity is not always solvable:** if you have no valid instrument, no panel data, and no proxy, you can only discuss the *likely direction of bias*.